In [0]:
# Load the data from the volume

catalog_name = 'workspace' # choose the catalog and the schema
schema_name = 'default'

business_path = "/Volumes/workspace/default/yelpdataset/yelp_academic_dataset_business.json"
review_path = "/Volumes/workspace/default/yelpdataset/yelp_academic_dataset_review.json"
user_path = "/Volumes/workspace/default/yelpdataset/yelp_academic_dataset_user.json"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [0]:
# Get an overview of the raw data loaded

business_raw = spark.read.json(business_path)
review_raw = spark.read.json(review_path)
user_raw = spark.read.json(user_path)

business_raw.printSchema()
review_raw.printSchema()
user_raw.printSchema()

display(business_raw)
display(review_raw)
display(user_raw)

business_raw.show(5)
review_raw.show(5)
user_raw.show(5)

In [0]:
# Bronze layer

business_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_business")
review_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_review")
user_raw.write.mode("overwrite").format("delta").saveAsTable("bronze_user")


In [0]:
# need to compare the business raw and the bronze business

bronze_business = spark.read.table("bronze_business") #assign a variable to read the table

business_raw.printSchema()
bronze_business.printSchema()

In [0]:
# silver layer

from pyspark.sql import functions as F

silver_business = business_raw.select("business_id", "name", "state", "stars", "review_count", "categories").withColumnRenamed("stars", "business_stars").withColumnRenamed("review_count", "business_review_count")

silver_user = user_raw.select("average_stars", "user_id", "review_count", "useful").withColumnRenamed("average_stars", "user_stars").withColumnRenamed("review_count", "user_review_count")

silver_review =review_raw.select("business_id", "stars", "date", "text", "user_id").withColumnRenamed("stars", "review_stars").withColumn("new_date", F.year(F.try_to_date(F.col("date"), "yyyy-MM-dd")))

silver_business.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_business")
silver_user.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_user")
silver_review.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_review")


In [0]:
# join review to business

review_business = silver_review.join(
    silver_business,
    on= "business_id",
    how="inner"
)

# join the above result to user

business_review_user = review_business.join(
    silver_user,
    on="user_id",
    how="inner"
)
# DBTITLE 1,write to delta
business_review_user.write.mode("overwrite").format("delta").saveAsTable("gold_business_review")

In [0]:
combination = spark.read.table("gold_business_review")
combination.show(5)
combination.printSchema()

# Gold layer


In [0]:
id_group = combination.groupBy("business_id")
id_group.count().orderBy("count", ascending=False).show(5)

In [0]:
id_group2 = combination.groupBy("business_id")
id_group2.avg("business_stars").orderBy("avg(business_stars)", ascending=False).show(5)

In [0]:
id_group3 = combination.groupBy("business_id")
id_group3.avg("review_stars").orderBy("avg(review_stars)", ascending=False).show(5)

In [0]:
group4 = combination.groupBy("state", "categories")
group4.count().orderBy("count", ascending=False).show(5)
